In [3]:
# LangChain RAG Exercise: Build a "Chat with Your Document" System
# =================================================================
# This notebook walks you through building a simple RAG system step-by-step
# SETUP: Run this first to install required packages
!pip install -U langchain langchain-community langchain-openai langchain-text-splitters chromadb pypdf

In [ ]:
# ============================================================================
# PART 1: SETUP AND IMPORTS
# ============================================================================

import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

# Load API key from .env
load_dotenv(r'C:\Users\Gurveer\ds-portfolio\.env')
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

print("✅ Setup complete! Libraries imported successfully.")


# ============================================================================
# PART 2: LOAD YOUR PDF DOCUMENT
# ============================================================================

pdf_path = r"C:\Users\Gurveer\Downloads\math2100-test2-review-questions-with-answers.pdf"

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print(f"✅ Loaded {len(documents)} pages from the PDF")
print(f"📄 First page preview:\n{documents[0].page_content[:500]}...")


# ============================================================================
# PART 3: SPLIT DOCUMENT INTO CHUNKS
# ============================================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,    
    chunk_overlap=100,
    length_function=len,
)

chunks = text_splitter.split_documents(documents)

print(f"✅ Split into {len(chunks)} chunks")
print(f"📝 First chunk preview:\n{chunks[0].page_content[:300]}...")


# ============================================================================
# PART 4: CREATE EMBEDDINGS + VECTOR DATABASE
# ============================================================================

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("✅ Vector database created successfully!")
print(f"📊 Total vectors stored: {vectorstore._collection.count()}")


# ============================================================================
# PART 5: TEST SIMILARITY SEARCH
# ============================================================================

test_query = "What is this document about?"

similar_docs = retriever.invoke(test_query)

print(f"\n🔍 Top chunks matching: '{test_query}'")
print("=" * 80)
for i, doc in enumerate(similar_docs, 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content[:300])
    print("...")


# ============================================================================
# PART 6: BUILD THE MODERN RAG Q&A SYSTEM (LCEL)
# ============================================================================

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

prompt_template = """
Use the context below to answer the question.
If the answer cannot be found in the context, say "I don't know."
Keep the answer under three sentences.

Context:
{context}

Question:
{question}

Answer:
"""

PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template=prompt_template
)

# ✅ Modern RAG pipeline (replaces RetrievalQA)
rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | PROMPT
    | llm
)

print("✅ Modern RAG system ready!")


# ============================================================================
# PART 7: INTERACTIVE Q&A
# ============================================================================

def ask_question(question):
    """Ask the RAG system a question and show results."""
    response = rag_chain.invoke(question)
    
    print("\n" + "=" * 80)
    print("❓ Question:", question)
    print("=" * 80)
    print("💡 Answer:", response.content)
    print("=" * 80)


questions = [
    "What is the main topic of this document?",
    "Can you summarize the key points?",
    "What are the most important takeaways?"
]

for q in questions:
    ask_question(q)


# ============================================================================
# BONUS: INTERACTIVE CHAT
# ============================================================================

def chat_with_document():
    print("\n🤖 Chat with Your Document (type 'quit' to exit)")
    print("=" * 80)

    while True:
        user_q = input("\nYour question: ").strip()
        if user_q.lower() in ("quit", "exit", "q"):
            print("👋 Goodbye!")
            break
        
        if not user_q:
            continue

        ask_question(user_q)

# chat_with_document()


# ============================================================================
# STUDENT EXERCISES
# ============================================================================

print("\n" + "=" * 80)
print("📝 STUDENT EXERCISES")
print("=" * 80)
print("""
1. EXPERIMENT WITH CHUNK SIZE
2. TRY DIFFERENT QUESTIONS
3. MODIFY THE PROMPT
4. EXPERIMENT WITH TEMPERATURE
5. TEST WITH DIFFERENT DOCUMENTS
6. ANALYZE RETRIEVAL QUALITY
7. ADD CONVERSATION MEMORY
8. CHALLENGE: COMPARE TWO PDFs
""")

# ============================================================================
# CLEAN UP (OPTIONAL)
# ============================================================================

# Uncomment to delete the vector database and start fresh
# import shutil
# shutil.rmtree("./chroma_db")
# print("🗑️ Vector database deleted")

print("\n✅ Notebook complete! Happy learning! 🎓")

✅ Setup complete! Libraries imported successfully.
✅ Loaded 2 pages from the PDF
📄 First page preview:
Math 2100, Test 2 practice questions 2025
1. Consider the table here: x P (x)
-3 .07
5 .12
7 .41
13 .23
23 .17
(a) Is this a valid PMF for an experiment with outcome x?
(b) What is P (x < 0)? Ans: 0.07
(c) What is P (7 ≤ x < 23)? Ans: 0.64
(d) What is P (x ≥ 25)? Ans: 0
(e) Expected value and variance of X. Ans: E[X]=10.16, Var(X)=152.52 − (10.16)2=49.2944
(f) expected value and variance of 2X-4 Ans: E[2X-4]=2E[X]-4=2(10.16)-4, Var(2X-4)=4Var(X)=4(49.2944)
(g) find the cumulant distributive func...
✅ Split into 13 chunks
📝 First chunk preview:
Math 2100, Test 2 practice questions 2025
1. Consider the table here: x P (x)
-3 .07
5 .12
7 .41
13 .23
23 .17
(a) Is this a valid PMF for an experiment with outcome x?
(b) What is P (x < 0)? Ans: 0.07
(c) What is P (7 ≤ x < 23)? Ans: 0.64
(d) What is P (x ≥ 25)? Ans: 0
(e) Expected value and varian...
✅ Vector database created successfully!
📊 T